# VGG19 — Pediatric Chest X-Ray Classification
**NIH Chest X-Ray14 | Transfer Learning | Multi-Label (14 Classes)**

| | |
|---|---|
| **Author** | Dania Beloufa |
| **GitHub** | [@daniabel90](https://github.com/daniabel90) |
| **Dataset** | [NIH Chest X-Ray14](https://www.kaggle.com/datasets/nih-chest-xrays/data) |
| **Architecture** | 19 weight layers (16 conv + 3 FC) |
| **Test Accuracy** | 86.8% |
| **Mean AUC** | 0.80 |

---

## Overview

Deeper VGG variant with 19 layers for enhanced feature extraction on chest X-ray pathologies.

Deeper than VGG16, VGG19 captures more abstract features at the cost of slightly higher computational demand.


## 1. Setup & Imports

In [ ]:
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

gpus = tf.config.list_physical_devices('GPU')
if gpus: tf.config.experimental.set_memory_growth(gpus[0], True)
print(f'TF {tf.__version__} | GPU: {len(gpus)>0}')
SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

## 2. Dataset Configuration

In [ ]:
DATA_DIR   = '/kaggle/input/nih-chest-xrays/data'
CSV_PATH   = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')
IMG_DIR    = os.path.join(DATA_DIR, 'images')
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 25
LR         = 1e-4

CLASSES = ['Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass',
           'Nodule','Pneumonia','Pneumothorax','Consolidation','Edema',
           'Emphysema','Fibrosis','Pleural_Thickening','Hernia']
print(f'{len(CLASSES)} classes: {CLASSES}')

## 3. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv(CSV_PATH)
for cls in CLASSES:
    df[cls] = df['Finding Labels'].apply(lambda x: 1 if cls in x.split('|') else 0)
df['image_path'] = df['Image Index'].apply(lambda x: os.path.join(IMG_DIR, x))
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Valid samples: {len(df)}')

plt.figure(figsize=(14,4))
df[CLASSES].sum().sort_values(ascending=False).plot(kind='bar',color='steelblue',edgecolor='black')
plt.title('VGG19 — Class Distribution', fontweight='bold')
plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.show()

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=SEED)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=10,
    width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, zoom_range=0.1)
val_datagen = ImageDataGenerator(rescale=1./255)

def make_gen(datagen, df, shuffle=True):
    return datagen.flow_from_dataframe(df, x_col='image_path', y_col=CLASSES,
        target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='raw', shuffle=shuffle)

train_generator = make_gen(train_datagen, train_df)
val_generator   = make_gen(val_datagen, val_df, shuffle=False)
test_generator  = make_gen(val_datagen, test_df, shuffle=False)

## 4. Build VGG19 Model

In [ ]:
from tensorflow.keras.applications import VGG19

base_model = VGG19(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(14, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

## 5. Compile & Train (Phase 1 — Feature Extraction)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(LR),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

callbacks = [
    ModelCheckpoint('vgg19_best.h5', monitor='val_auc', mode='max', save_best_only=True),
    EarlyStopping(monitor='val_auc', patience=7, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7)
]

history = model.fit(train_generator, validation_data=val_generator,
                    epochs=EPOCHS, callbacks=callbacks, verbose=1)

## 6. Fine-Tuning (Phase 2)

In [ ]:
base_model.trainable = True
freeze_until = int(len(base_model.layers) * 0.6)
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

history_ft = model.fit(train_generator, validation_data=val_generator,
                       epochs=10, callbacks=callbacks)

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (m, lbl) in zip(axes, [('loss','Loss'),('accuracy','Accuracy'),('auc','AUC')]):
    ax.plot(history.history[m], lw=2, label='Train P1')
    ax.plot(history.history[f'val_{m}'], lw=2, ls='--', label='Val P1')
    off = len(history.history[m])
    ax.plot(range(off, off+len(history_ft.history[m])), history_ft.history[m], lw=2, label='Train P2')
    ax.plot(range(off, off+len(history_ft.history[f'val_{m}'])), history_ft.history[f'val_{m}'], lw=2, ls='--', label='Val P2')
    ax.set_title(f'VGG19 — {lbl}', fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluation — Per-Class AUC

In [ ]:
test_generator.reset()
y_pred = model.predict(test_generator, verbose=1)
y_true = test_df[CLASSES].values

auc_scores = {}
print('Per-Class AUC:\n')
for i, cls in enumerate(CLASSES):
    try:
        auc = roc_auc_score(y_true[:,i], y_pred[:,i])
        auc_scores[cls] = auc
        print(f'  {cls:25s}: {auc:.4f}')
    except: pass
print(f'\nMean AUC: {np.mean(list(auc_scores.values())):.4f}')

y_bin = (y_pred>=0.5).astype(int)
print(classification_report(y_true, y_bin, target_names=CLASSES, zero_division=0))

## 9. ROC Curves (14 Pathologies)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()
for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_true[:,i], y_pred[:,i])
    axes[i].plot(fpr, tpr, lw=2, label=f'AUC={auc_scores.get(cls,0):.3f}')
    axes[i].plot([0,1],[0,1],'k--',lw=1)
    axes[i].set_title(cls, fontweight='bold')
    axes[i].legend(loc='lower right'); axes[i].grid(alpha=0.3)
for j in range(len(CLASSES), len(axes)): axes[j].set_visible(False)
plt.suptitle('VGG19 — ROC Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Grad-CAM Visualization

In [ ]:
import cv2

def grad_cam(model, img_array, layer_name, class_idx):
    grad_model = tf.keras.Model([model.inputs],
        [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0)
    return (heatmap / (tf.math.reduce_max(heatmap)+1e-8)).numpy()

sample = test_df['image_path'].iloc[0]
img = tf.keras.preprocessing.image.load_img(sample, target_size=(224,224))
arr = np.expand_dims(tf.keras.preprocessing.image.img_to_array(img)/255.0, 0)
pred = model.predict(arr)[0]
top_idx = np.argmax(pred)
print(f'Predicted: {CLASSES[top_idx]} ({pred[top_idx]:.3f})')

try:
    last_conv = [l.name for l in model.layers if 'conv' in l.name.lower()][-1]
    hm = grad_cam(model, arr, last_conv, top_idx)
    hm_r = cv2.resize(hm, (224,224))
    hm_c = cv2.applyColorMap(np.uint8(255*hm_r), cv2.COLORMAP_JET)
    orig = np.uint8(arr[0]*255)
    ov = cv2.addWeighted(orig, 0.6, hm_c, 0.4, 0)
    fig, axes = plt.subplots(1,3,figsize=(15,5))
    for ax, im, t in zip(axes, [orig, hm_r, ov],
        ['Original X-Ray', f'Grad-CAM ({CLASSES[top_idx]})', 'Overlay']):
        ax.imshow(im if im.ndim==3 else im, cmap=None if im.ndim==3 else 'jet')
        ax.set_title(t, fontweight='bold'); ax.axis('off')
    plt.tight_layout()
    plt.savefig('gradcam.png', dpi=150)
    plt.show()
except Exception as e: print(f'Grad-CAM: {e}')

## 11. Results Summary

| Metric | Score |
|--------|-------|
| **Test Accuracy** | **86.8%** |
| **Mean AUC** | **0.80** |
| **Macro F1** | **0.73** |
| Architecture | 19 weight layers (16 conv + 3 FC) |
| Dataset | NIH Chest X-Ray14 (112,120 images) |
| Input | 224×224×3 |
| Classes | 14 chest pathologies |

> *From: "Comparative Analysis of CNN Architectures for Pediatric Chest X-Ray Classification" — Dania Beloufa, Université Saad Dahleb Blida 1, 2025*

In [ ]:
model.save('vgg19_chestxray_final.h5')
with open('classes.json','w') as f: json.dump(CLASSES, f, indent=2)
print('Saved: vgg19_chestxray_final.h5 + classes.json')